# Generating Displaced Coordinates from Survey Data
This notebook serves two purposese as follows:
1. **Usage example.** A walkthough of how to use the ```point_displacement``` module to generate 
displaced coordinates from household survey GPS coordinates data.
2. **Benchmark results from ```point_displacement```.** Use results from the ArcGIS version of the process to test whether the results from ```point_displacement``` are correct.

## Python setup

In [1]:
import os
import sys
import random
import math
import numpy as np
from pathlib import Path
import pandas as pd
import geopandas as gpd
from pyproj import CRS, Proj,Geod
from pyproj import Transformer
from shapely import geometry
from datetime import datetime


SRC = Path.cwd().parents[0].joinpath('src')
sys.path.insert(1, str(SRC))
import point_displacement 

## Input/output data setup
Provide absolute paths to the folliwing files:
1. **GPS coordinates file.** This is the file with GPS coordinates which need to be displaced. For now, we are supporting only **CSV** and **STATA** ```.dta``` file formats.
2. **Administrative regions.**  A shapefile (or other geographic formats such as Geojson) of admin regions within which the displacements will be confined.
3. **outputs directory.** Provide directory where outputs from running the processing will be saved.

In [2]:
# base data folder
DATA = Path.cwd().parents[0].joinpath('data')

# ========================
# INPUT FILES FOR ETHIOPIA
# ========================
ETH_SURVEY = DATA.joinpath('ETH', 'survey')
ETH_GEOVARS = DATA.joinpath('ETH', 'w4_geovars')

# GPS coordinates file
ETH_COORDS = ETH_SURVEY.joinpath("HH level coordinates.dta")

# Admin regions (adm2) shapefile
ETH_ADM = ETH_GEOVARS.joinpath("spatial_data", "shp_adm2", "eth_admbnda_adm2_csa_bofed_20190827.shp")

# output directory
ETH_OUPUT_DIR = DATA.joinpath("ETH", "outputs")

## Parameters and other global variables setup
Now, we setep parameters as well as helper global variables as follows:
### Displacement parameters
All distance paramters are in meters
1. **urb_max_dist.** The maximum distance to displace a point in urban areas.
2. **rur_max_dist.** The maximum distance to displace a point in rural areas.
3. **rur_max_dist_subset.** The maximum distance to displace a point in rural areas for a subset of the households. 
4. **urb_min_dist.** Allowed minimum distance between original point and displaced point in urban areas.
5. **rur_min_dist.** Allowed minimum distance between original point and displaced point in rural areas.
6. **apply_min_dist.** whether to apply minimum distance threshold or not
7. **rur_subset_prop.** proportion of rural households to apply special maximum distance

### Processing variables/args 
1. **geodetic_crs_epsg.** The geographic CRS for which the latitude and longitude were collected. The default is WGS84 given as EPSG code. If you have the datum-name or CRS details but do not know the EPSG code, please use [epsg.io](https://epsg.io)
2. **lon_col.** Column name containing longitude
3. **lat_col.** Column name containing latitude
4. **uniq_id.** Column name (if available uniquely identifying each row
5. **urb_rur_col.** Column name indicating whether point is in urban or rural
6. **max_tries.** In some cases where there are many coordinates and the confining polygon is small, finding displaced coordinates withing the polygon takes too long and we give up after these many tries.

In [68]:
# =====================
# SETUP PARAMS 
# ====================
# urb_max_dist
urb_max = 2000
# rur_max_dist
rur_max = 5000
# rur_max_dist_subset
rur_max_subset = 10000
# urb_min_dist
urb_min = 100
# rur_min_dist
rur_min = 1000
# apply_min_dist
apply_min_dist = False
# rur_subset_prop = 10%
rur_sub_prop = 0.1
# max_tries
max_tries = 30

# setup the params dict
PARAMS = {'urb_max_dist': urb_max, 'rur_max_dist': rur_max, 'rur_max_dist_subset':rur_max_subset,
          'urb_min_dist': urb_min, 'rur_min_dist': rur_min, 'maximum_tries': max_tries,
          'apply_min_dist': apply_min_dist, 'rur_sub_prop': rur_sub_prop}



# ========================
# PREPARE PROCESSING ARGS
# =======================
# add EPSG code for geographic CRS if its not 
# default to WGS84
geo_crs = 4326
lon ='saq19__Longitude'
lat ='saq19__Latitude'
hhid = 'household_id'
urb_rur = 'rural_urban'
rural_code = 1

## Run the displacement

In [4]:
# Create displacer object
eth_displacer = point_displacement.SimpleDisplacer(PARAMS, input_coords=ETH_COORDS,  
                                               shp_adm_bounds=ETH_ADM, output_dir=ETH_OUPUT_DIR,
                                               lon_col=lon, lat_col=lat, urb_rur_col=urb_rur, 
                                                   unique_id=hhid, rural_code=rural_code, 
                                                   geodetic_crs=geo_crs)

# Run the displacer
eth_displacer.run_displacer() 

PREPARING DATA FOR DISPLACEMENT...........
################################################## 


132 null lat/lon rows dropped out of 4,959

Dropped observations saved to file: 
 /Users/dmatekenya/WBG-LOCAL/Spatial-Anonymization/data/ETH/outputs/HH level coordinates_dropped.csv
Found the following projected CRS: ['32637', '32636', '32638']


/Users/dmatekenya/WBG-LOCAL/Spatial-Anonymization/src/point_displacement.py:341: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hh_coords[self.proj_crs_col] = df_hh_coords.apply(lambda x:


ADDING DISPLACED POINTS...........
################################################## 

RUNNING SOME CHECKS .........
################################################## 

SAVING OUTPUTS .........
################################################## 


STATISTICS BASED OFF ORIGINAL VS. DISPLACED COORDS
      stat   area     value
0     mean  urban  0.977024
1     mean  rural  2.656802
2   median  urban  0.956274
3   median  rural  2.544150
4  minimum  urban  0.000960
5  minimum  rural  0.004040
6  maximum  urban  1.999770
7  maximum  rural  9.849440

Summary statistics saved to: 
 /Users/dmatekenya/WBG-LOCAL/Spatial-Anonymization/data/ETH/outputs/HH level coordinates_disp_stats.csv

----------------- DONE WITH PROCESSING ------------------- 

Processing 4,827 coordinates with 30 maximum tries per coordinate took 0 minutes


## Compute geospatial variables based off of pertubed coordinates

In [42]:
# Use this as reference file from Siobhan's pipeline
df_ref = pd.read_stata("../data/ETH/w4_geovars/puf/ETH_HouseholdGeovariables_Y4.dta", 
                   convert_categoricals=False)
df_ref['household_id'] = df_ref['household_id'].astype(str)
df_ref.shape

(6804, 42)

In [40]:
# Load data from Python pipeline
results_from_python = ETH_OUPUT_DIR.joinpath('HH level coordinates_disp.csv')
df_py = pd.read_csv(results_from_python)

# deal with missing trailing zero in the HH id
df_py['household_id'] = df_py['household_id'].apply(lambda x: '0{}'.format(x) if len(str(x)) == 17 else str(x))
# dfc  = dfc[['household_id', 'saq19__Latitude', 'saq19__Longitude']]

In [45]:
# Lets use only those HH which match
# The mismatch could be explained by several reasons as below:
# 1. The input file I used has 4,959 HHs as opposed to 6, 804 in the reference file
# 2. During my processing, I dropped 132 HH with null lat/lon values
df_check =  df_ref.merge(df_py, on='household_id', how='inner')
print('Number of matched HHs: {:,}'.format(len(df_check)))

Number of matched HHs: 4,675


In [69]:
    def generate_displacement_stats(df_res, stats=None):
        """
        Run diagnostic summary statistics on distance between original
        and displaced point
        Parameters
        ----------
        stats - List of summary statistics to run. For example, mean, median, min, maximamum

        Returns Saves summary statistics as a CSV file
        -------

        """
        # ===================================
        # CALCULATE DISTANCES
        # ===================================
        print(lon, lat)
        df_res['dist'] = df_res.apply(lambda x:
                                          point_displacement.distance_latlon_points((x[lon], x[lat]),
                                                                 (x['lon_mod'], x['lat_mod'])), axis=1)

        
        return df_res
        # ===================================
        # GET SUMMARY STATISTICS
        # ===================================
        df_rur = df_res.query('{} == {}'.format(urb_rur, rural_code))
        assert len(df_res[urb_rur].unique()) == 2, 'THERE SHOULD BE ONLY TWO URBAN/RURAL VALUES'
        urb_code = list(set(df_res[urb_rur].unique()) - {rural_code})[0]
        df_urb = df_res.query('{} == {}'.format(urb_rur, urb_code))

        res = []
        if 'mean' in stats:
            urb_mean = df_urb['dist'].mean()
            rur_mean = df_rur['dist'].mean()
            res.append({'stat': 'mean', 'area': 'urban', 'value': urb_mean})
            res.append({'stat': 'mean', 'area': 'rural', 'value': rur_mean})
        if 'median' in stats:
            urb_med = df_urb['dist'].median()
            rur_med = df_rur['dist'].median()
            res.append({'stat': 'median', 'area': 'urban', 'value': urb_med})
            res.append({'stat': 'median', 'area': 'rural', 'value': rur_med})

        if 'min' in stats or 'minimum' in stats:
            urb_min = df_urb['dist'].min()
            rur_min = df_rur['dist'].min()
            res.append({'stat': 'minimum', 'area': 'urban', 'value': urb_min})
            res.append({'stat': 'minimum', 'area': 'rural', 'value': rur_min})

        if 'max' in stats or 'maximum' in stats:
            urb_max = df_urb['dist'].max()
            rur_max = df_rur['dist'].max()
            res.append({'stat': 'maximum', 'area': 'urban', 'value': urb_max})
            res.append({'stat': 'maximum', 'area': 'rural', 'value': rur_max})

        # ===================================
        # PRINT AND SAVE SUMMARY STATISTICS
        # ===================================
        df_stats = pd.DataFrame(res)
        print()
        print("="*50)
        print('STATISTICS BASED OFF ORIGINAL VS. DISPLACED COORDS')
        print("=" * 50)
        print(df_stats)

#         output_csv_filename = "{}_disp_stats.csv".format(self.data_file.parts[-1].split(".")[0])
#         output_csv_fpath = self.output_dir.joinpath(output_csv_filename)

#         print()
#         print('Summary statistics saved to: \n {}'.format(output_csv_fpath))
#         df_stats.to_csv(output_csv_fpath, index=False)
        return df_stats

In [70]:
df_ref_stats = generate_displacement_stats(df_res=df_check, stats=['mean', 'median', 'max', 'min'])

saq19__Longitude saq19__Latitude


In [74]:
df_ref_stats[['household_id', 'saq19__Longitude', 'saq19__Latitude', 'lon_mod', 'lat_mod', 'rural_urban', 'dist']]

,household_id,saq19__Longitude,saq19__Latitude,lon_mod,lat_mod,rural_urban,dist
0,020101010100104011,41.072967,11.739229,41.080364,11.749938,2,1.432994
1,020101010100104022,41.073753,11.738610,41.080364,11.749938,2,1.445549
2,020101010100104033,41.073781,11.738010,41.080364,11.749938,2,1.501942
3,020101010100104044,41.074661,11.737787,41.080364,11.749938,2,1.480925
4,020101010100104055,41.075414,11.737486,41.080364,11.749938,2,1.479348
...,...,...,...,...,...,...,...
4670,150101088803701057,41.660067,9.510286,41.663799,9.504422,1,0.767203
4671,150101088803701067,41.660191,9.510225,41.663799,9.504422,1,0.754227
4672,150101088803701077,41.660088,9.510192,41.663799,9.504422,1,0.757179
4673,150101088803701087,41.660042,9.510244,41.663799,9.504422,1,0.764759


In [ ]:
"020101010100104011"

In [ ]:
df_ref.household_id.nunique()

In [ ]:
# Since the reference file from Siobhan has more observations
# Lets keep only the households in common
df_check = df_ref_coords.merge(df_py, on='household_id', how='inner', indicator=True)

In [ ]:
df_check._merge.value_counts()

In [ ]:
df[['household_id', 'lat_mod', 'lon_mod']]

In [ ]:
df['s'] = df.household_id.apply(lambda x: 1 if x in selected_ids else 0)

In [ ]:
df['s'].value_counts()

In [ ]:
type(selected_ids)

In [ ]:
selected_ids[0]

In [ ]:
130102088800104058 in selected_ids

In [ ]:
list(set(df.rural_urban.unique()) - set([2]))[0]

In [ ]:
assert len(df.rural_urban.unique()) == 2

In [ ]:
df.columns.difference(x)

In [ ]:
x = ['Shape_Leng', 'Shape_Area', 'ADM2_EN', 'ADM2_PCODE', 'ADM2_REF',
       'ADM2ALT1EN', 'ADM2ALT2EN', 'ADM1_EN', 'ADM1_PCODE', 'ADM0_EN']

In [32]:
"20101010100104011".zfill(2)

'20101010100104011'

In [33]:
str.zfill?

Signature: str.zfill(self, width, /)
Docstring:
Pad a numeric string with zeros on the left, to fill a field of the given width.

The string is never truncated.
Type:      method_descriptor
